In [ ]:
import subprocess, sys, traceback, os

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"$ {' '.join(cmd)}\n{r.stdout[-3000:]}\n{r.stderr[-3000:]}", flush=True)
    r.check_returncode()
    return r

try:
    ckpt("STEP0: /kaggle/input listing")
    for root, dirs, files in os.walk("/kaggle/input"):
        ckpt(f"  {root}: dirs={dirs} files={files[:10]}")

    ckpt("STEP1: pip install pytorch_tabular")
    run([sys.executable, "-m", "pip", "install", "-q", "pytorch_tabular"])

    ckpt("STEP1: git clone")
    subprocess.run(["rm", "-rf", "kaggle_playground_s6e9"])
    run(["git", "clone", "-q", "https://github.com/gccarno/kaggle_playground_s6e9.git"])

    import torch
    ckpt(f"STEP1: cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        ckpt(f"STEP1: device: {torch.cuda.get_device_name(0)}")
        ckpt(f"STEP1: mem GB: {torch.cuda.get_device_properties(0).total_memory / 1e9}")
    ckpt("STEP1: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP1 FAILED\n" + tb)
    raise

In [ ]:
import json, os, sys, time, math, traceback

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

# ModernNCA single-fold diagnostic (Ye et al., ICLR 2025). Adapted from the S6E8
# implementation (gcarno/s6e8-mnca) to this repo's simpler TE representation -- a plain
# MLP encoder rather than S6E8's per-value token embeddings, since the retrieval
# mechanism itself (soft-kNN in a learned metric space, self-retrieval masked,
# candidates resampled per step) is what's being tested, not S6E8's specific tokenizer.
# Skips the test set entirely, same reasoning as P1 (TabICL): this is an architecture
# question, not a submission candidate.
try:
    ckpt("STEP2: importing pipeline.py")
    sys.path.insert(0, "kaggle_playground_s6e9/src")
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    import pipeline as pl

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt(f"STEP2: device {dev}")

    CFG = {
        **pl.DEFAULTS,
        "te_cols": ["Annual_Income_USD", "Daily_Commute_km", "Age"],
        "te_smooth": 5.0,
        "te_backoff": "neighborhood",
    }
    N_SUB = 300000
    EPOCHS = 15

    train = pd.read_csv(pl.DATA_DIR / "train.csv")
    test = pd.read_csv(pl.DATA_DIR / "test.csv")
    y = (train[pl.TARGET] == "Yes").astype(int).values
    train, test, feats_num, feats_cat = pl.base_features(train, test, CFG)
    feats = feats_num + feats_cat

    skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=CFG["cv_seed"])
    i, j = next(iter(skf.split(train[feats], y)))
    Xtr, Xva = train[feats].iloc[i].copy(), train[feats].iloc[j].copy()
    Xte_unused = test[feats].copy()
    ytr_full, yva = y[i], y[j]

    ckpt("STEP2: fitting target encoders on fold 0")
    pl.apply_target_encoding(Xtr, ytr_full, Xva, Xte_unused, CFG["te_cols"], CFG, CFG["cv_seed"])
    ckpt(f"STEP2: features = {list(Xtr.columns)}")

    A = pd.get_dummies(Xtr, columns=feats_cat, drop_first=True).astype(np.float32)
    B = pd.get_dummies(Xva, columns=feats_cat, drop_first=True).astype(np.float32).reindex(
        columns=A.columns, fill_value=0)
    mu, sd = A.values.mean(0), A.values.std(0) + 1e-6
    Ad = (A.values - mu) / sd
    Bd = (B.values - mu) / sd

    idx = np.random.default_rng(42).choice(len(Ad), size=min(N_SUB, len(Ad)), replace=False)
    Xs = Ad[idx].astype(np.float32)
    ys = ytr_full[idx].astype(np.float32)
    Xv = Bd.astype(np.float32)
    n_feat = Xs.shape[1]
    ckpt(f"STEP2: training pool = {len(Xs)} rows (of {len(Ad)} available), {n_feat} features, "
         f"{EPOCHS} epochs")

    class Encoder(nn.Module):
        def __init__(self, d_in, d_hidden=256, d_out=64, p_drop=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d_in, d_hidden), nn.BatchNorm1d(d_hidden), nn.ReLU(), nn.Dropout(p_drop),
                nn.Linear(d_hidden, d_out),
            )

        def forward(self, x):
            return self.net(x)

    NEG = -1e30

    def _lse(m, s, blk):
        nm = torch.maximum(m, blk.max(1).values)
        e = torch.where(blk > NEG / 2, (blk - nm.unsqueeze(1)).exp(), torch.zeros_like(blk))
        return nm, s * torch.exp(m - nm) + e.sum(1)

    def nca_logp(zq, zc, yc, mask=None):
        sc = -torch.cdist(zq, zc)
        if mask is not None:
            sc = sc.masked_fill(mask, NEG)
        yb = yc.unsqueeze(0) > 0.5
        allz = torch.logsumexp(sc, 1)
        pos = torch.logsumexp(torch.where(yb, sc, torch.full_like(sc, NEG)), 1)
        neg = torch.logsumexp(torch.where(yb, torch.full_like(sc, NEG), sc), 1)
        return pos - allz, neg - allz

    @torch.no_grad()
    def nca_predict(net, Xq, ZC, YC, qchunk=2048, cchunk=32768):
        net.eval()
        out = []
        for k in range(0, len(Xq), qchunk):
            zq = net(torch.from_numpy(Xq[k:k + qchunk]).to(dev))
            n = len(zq)
            mA = torch.full((n,), NEG, device=dev); sA = torch.zeros(n, device=dev)
            mP = torch.full((n,), NEG, device=dev); sP = torch.zeros(n, device=dev)
            for c in range(0, len(ZC), cchunk):
                sc = -torch.cdist(zq, ZC[c:c + cchunk])
                yb = YC[c:c + cchunk].unsqueeze(0) > 0.5
                mA, sA = _lse(mA, sA, sc)
                mP, sP = _lse(mP, sP, torch.where(yb, sc, torch.full_like(sc, NEG)))
            out.append(torch.exp((mP + sP.log()) - (mA + sA.log())).float().cpu().numpy())
        return np.concatenate(out)

    net = Encoder(n_feat).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-5)
    ntr = len(Xs)
    batch, cand = 512, min(8192, ntr - 1)
    nstep = math.ceil(ntr / batch) * EPOCHS + 10
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=nstep, pct_start=0.15)
    Xs_t = torch.from_numpy(Xs).to(dev)
    ys_t = torch.from_numpy(ys).to(dev)

    best_auc, best_w, best_ep, since, patience = -1.0, None, 0, 0, 4
    best_p = None
    t0 = time.time()
    for ep in range(EPOCHS):
        net.train()
        perm = torch.randperm(ntr, device=dev)
        for s in range(0, ntr, batch):
            sl = perm[s:s + batch]
            cd = torch.randint(0, ntr, (cand,), device=dev)
            zq, zc = net(Xs_t[sl]), net(Xs_t[cd])
            lp1, lp0 = nca_logp(zq, zc, ys_t[cd], sl.unsqueeze(1) == cd.unsqueeze(0))
            yq = ys_t[sl]
            loss = -(yq * lp1 + (1 - yq) * lp0).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step(); sched.step()

        with torch.no_grad():
            ZC = net(Xs_t)
        p = nca_predict(net, Xv, ZC, ys_t)
        a = roc_auc_score(yva, p)
        ckpt(f"  epoch {ep:2d}  val AUC {a:.6f}  ({time.time() - t0:.1f}s elapsed)")
        if a > best_auc:
            best_auc, best_ep, since, best_p = a, ep, 0, p
        else:
            since += 1
            if since >= patience:
                ckpt("  early stop")
                break

    metrics = {
        "run_tag": "MNCA1", "learner": "mnca_diag", "final_oof_auc": round(float(best_auc), 6),
        "n_train": ntr, "n_val": len(Xv), "best_epoch": best_ep,
        "wall_sec": round(time.time() - t0, 1),
    }
    ckpt(f"STEP2: fold-0 AUC = {best_auc:.6f}  (n={ntr}, val={len(Xv)}, best_ep={best_ep})")
    print("RUN_METRICS_JSON:" + json.dumps(metrics), flush=True)

    np.save("/kaggle/working/mnca_fold0_proba.npy", best_p)
    np.save("/kaggle/working/mnca_fold0_idx.npy", j)
    ckpt("STEP2: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP2 FAILED\n" + tb)
    raise

In [ ]:
import json, os, sys, time, traceback

CKPT = "/kaggle/working/checkpoint.txt"

def ckpt(msg):
    print(msg, flush=True)
    with open(CKPT, "a") as f:
        f.write(msg + "\n")

# TabTransformer (pytorch_tabular) single-fold diagnostic, same protocol as every other
# architecture this session: E1's exact TE representation, real frozen fold-0 split,
# full validation fold. Locally this hung deterministically on GPU/Lightning startup
# variance; running on Kaggle's clean environment instead of continuing to fight that.
try:
    ckpt("STEP3: importing pipeline.py + pytorch_tabular")
    sys.path.insert(0, "kaggle_playground_s6e9/src")
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    from pytorch_tabular import TabularModel
    from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
    from pytorch_tabular.models import TabTransformerConfig
    import pipeline as pl
    import torch

    CFG = {
        **pl.DEFAULTS,
        "te_cols": ["Annual_Income_USD", "Daily_Commute_km", "Age"],
        "te_smooth": 5.0,
        "te_backoff": "neighborhood",
    }
    N_SUB = 150000
    N_EPOCHS = 25

    train = pd.read_csv(pl.DATA_DIR / "train.csv")
    test = pd.read_csv(pl.DATA_DIR / "test.csv")
    y = (train[pl.TARGET] == "Yes").astype(int).values
    train, test, feats_num, feats_cat = pl.base_features(train, test, CFG)
    feats = feats_num + feats_cat

    skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=CFG["cv_seed"])
    i, j = next(iter(skf.split(train[feats], y)))
    Xtr, Xva = train[feats].iloc[i].copy(), train[feats].iloc[j].copy()
    Xte_unused = test[feats].copy()
    ytr, yva = y[i], y[j]

    pl.apply_target_encoding(Xtr, ytr, Xva, Xte_unused, CFG["te_cols"], CFG, CFG["cv_seed"])
    ckpt(f"STEP3: features = {list(Xtr.columns)}")

    num_cols = feats_num + ["te_Annual_Income_USD", "te_Daily_Commute_km", "te_Age"]
    cat_cols = feats_cat
    for c in num_cols:
        Xtr[c] = Xtr[c].astype("float64")
        Xva[c] = Xva[c].astype("float64")
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype(str)
        Xva[c] = Xva[c].astype(str)

    idx = np.random.default_rng(42).choice(len(Xtr), size=min(N_SUB, len(Xtr)), replace=False)
    train_df = Xtr.iloc[idx].copy()
    train_df["target"] = ytr[idx]
    val_df = Xva.copy()
    val_df["target"] = yva
    ckpt(f"STEP3: training on {len(train_df)} rows (of {len(Xtr)} available), {N_EPOCHS} epochs")

    accel = "gpu" if torch.cuda.is_available() else "cpu"
    data_config = DataConfig(target=["target"], continuous_cols=num_cols, categorical_cols=cat_cols)
    trainer_config = TrainerConfig(auto_lr_find=False, batch_size=1024, max_epochs=N_EPOCHS,
                                   accelerator=accel, progress_bar="none")
    optimizer_config = OptimizerConfig()
    model_config = TabTransformerConfig(task="classification", input_embed_dim=16,
                                        num_heads=4, num_attn_blocks=2)

    tabular_model = TabularModel(data_config=data_config, model_config=model_config,
                                 optimizer_config=optimizer_config, trainer_config=trainer_config)
    t0 = time.time()
    tabular_model.fit(train=train_df, validation=val_df)
    ckpt(f"STEP3: fit time {time.time() - t0:.1f}s")

    pred_df = tabular_model.predict(val_df)
    p = pred_df["target_1_probability"].values
    auc = roc_auc_score(yva, p)
    metrics = {
        "run_tag": "TABTFMR1", "learner": "tabtransformer_diag", "final_oof_auc": round(float(auc), 6),
        "n_train": len(train_df), "n_val": len(val_df), "wall_sec": round(time.time() - t0, 1),
    }
    ckpt(f"STEP3: fold-0 AUC = {auc:.6f}  (n={len(train_df)}, val={len(val_df)})")
    print("RUN_METRICS_JSON:" + json.dumps(metrics), flush=True)

    np.save("/kaggle/working/tabtransformer_fold0_proba.npy", p)
    np.save("/kaggle/working/tabtransformer_fold0_idx.npy", j)
    ckpt("STEP3: done")
except Exception:
    tb = traceback.format_exc()
    print(tb, flush=True)
    with open(CKPT, "a") as f:
        f.write("STEP3 FAILED\n" + tb)
    raise